In [ ]:
import os
import tensorflow as tf

In [ ]:
# Custom function to replace librosa.stft
def stft(y):
    return tf.signal.stft(y, frame_length=1024, frame_step=512, pad_end=False)

In [ ]:
class Model(tf.Module):
    @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
    def create_cnn_data(self, raw_data):
        Zxx = stft(raw_data)
        stft_sample = tf.abs(Zxx)
        return stft_sample

In [ ]:
model = Model()

converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [model.create_cnn_data.get_concrete_function()], model
)

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()

with open("stft.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: /tmp/tmpgpbowgak/assets


INFO:tensorflow:Assets written to: /tmp/tmpgpbowgak/assets
W0000 00:00:1709534487.488255   31152 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1709534487.488263   31152 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2024-03-04 12:11:27.488351: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgpbowgak
2024-03-04 12:11:27.488556: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2024-03-04 12:11:27.488560: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpgpbowgak
2024-03-04 12:11:27.490493: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2024-03-04 12:11:27.493918: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at path: /tmp/tmpgpbowgak
2024-03-04 12:11:27.496613: I tensorflow/cc/saved_model/loader.cc:317] SavedModel load for tags { serve }; Status: success: OK. Took 8263 m

In [ ]:
!echo "const unsigned char model[] = {" > model.h
!cat stft.tflite | xxd -i      >> model.h
!echo "};"                              >> model.h

model_h_size = os.path.getsize("model.h")
print(f"Header file, model.h, is {model_h_size:,} bytes.")

Header file, model.h, is 59,456 bytes.


In [ ]:
!sudo apt-get update && sudo apt-get -qq install xxd

In [ ]:
MODEL_TFLITE = '_________________.tflite' #enter the name of your TFlite file uploaded to the folders section
MODEL_TFLITE_MICRO = '_______________.cc' #update the name of your .cc file (This can be anything)
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}